# Análise de erros

Todo modelo erra. A questão é entender *onde* e *por quê*. Se o modelo erra sistematicamente num tipo de música, quem for usar o app precisa saber — senão toma decisão cega.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/tracks_clean.csv")

FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "duration_ms",
]

X = df[FEATURES]
y = df["is_hit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

xgb = XGBClassifier(
    n_estimators=200, max_depth=6,
    learning_rate=0.1, eval_metric="logloss", random_state=42,
)
xgb.fit(X_train, y_train)
print("Modelo treinado.")

Modelo treinado.


## Classificando cada predição

Separo o resultado do teste em quatro categorias: acertos nos hits, acertos nos não-hits, falsos positivos (achou que era hit mas não era) e falsos negativos (era hit e o modelo perdeu).

In [2]:
results = df.loc[X_test.index].copy()
results["pred"] = xgb.predict(X_test)
results["prob"] = xgb.predict_proba(X_test)[:, 1]


def classify_error(row):
    if row["is_hit"] == 1 and row["pred"] == 1:
        return "acerto_hit"
    if row["is_hit"] == 0 and row["pred"] == 0:
        return "acerto_nao_hit"
    if row["is_hit"] == 0 and row["pred"] == 1:
        return "falso_positivo"
    return "falso_negativo"


results["tipo"] = results.apply(classify_error, axis=1)
print(results["tipo"].value_counts().to_string())

tipo
acerto_nao_hit    14485
falso_negativo     1752
falso_positivo       17
acerto_hit           15


## Falsos positivos

Músicas que o modelo achou que seriam hit, mas não são. O que elas têm que enganou o modelo?

In [3]:
fp = results[results["tipo"] == "falso_positivo"]
hit_means = df[df["is_hit"] == 1][FEATURES].mean()

print(f"Total de falsos positivos: {len(fp)}\n")
print("Features dos falsos positivos acima da média dos hits:\n")
fp_means = fp[FEATURES].mean()
above = fp_means[fp_means > hit_means]
for feat in above.index:
    print(f"  {feat}: FP={fp_means[feat]:.4f}  hits={hit_means[feat]:.4f}")

Total de falsos positivos: 17

Features dos falsos positivos acima da média dos hits:

  acousticness: FP=0.6387  hits=0.2870
  instrumentalness: FP=0.5826  hits=0.0842


In [4]:
top5_fp = fp.nlargest(5, "prob")[["name", "artist", "popularity", "prob", "track_genre"]]
print("Top 5 falsos positivos (maior probabilidade prevista):")
top5_fp

Top 5 falsos positivos (maior probabilidade prevista):


,name,artist,popularity,prob,track_genre
73103,A Waiting,Even Massi,59,0.734712,sleep
3208,Filma Solo,Gabríel Ólafs,58,0.713177,ambient
73007,Pacific,Savannah De Lucia,59,0.675417,sleep
46087,Mixed Feelings,Soheil Soheili,0,0.650175,iranian
73011,Lejano,Pearl Melendez,55,0.600053,sleep


## Falsos negativos

Esses são os piores pro negócio: hits reais que o modelo não identificou. Oportunidades perdidas.

In [5]:
fn = results[results["tipo"] == "falso_negativo"]

print(f"Total de falsos negativos: {len(fn)}\n")
print("Perfil médio dos falsos negativos vs hits reais:\n")
comparison = pd.DataFrame({
    "falso_negativo": fn[FEATURES].mean(),
    "hits_reais": hit_means,
    "diferenca": fn[FEATURES].mean() - hit_means,
}).round(4)
comparison

Total de falsos negativos: 1752

Perfil médio dos falsos negativos vs hits reais:



,falso_negativo,hits_reais,diferenca
danceability,0.5953,0.5912,0.0041
energy,0.6410,0.6321,0.0089
valence,0.4853,0.4765,0.0088
tempo,121.4195,120.4051,1.0144
loudness,-7.6372,-7.8790,0.2417
speechiness,0.0810,0.0801,0.0009
acousticness,0.2758,0.2870,-0.0112
instrumentalness,0.0809,0.0842,-0.0033
duration_ms,219649.4144,217894.4442,1754.9702


In [6]:
top5_fn = fn.nsmallest(5, "prob")[["name", "artist", "popularity", "prob", "track_genre"]]
print("Top 5 falsos negativos (menor probabilidade prevista):")
top5_fn

Top 5 falsos negativos (menor probabilidade prevista):


,name,artist,popularity,prob,track_genre
3002,On the Nature of Daylight,Max Richter;Louisa Fuller;Natalia Bonner;John ...,65,0.002745,ambient
2945,Experience,Ludovico Einaudi;Daniel Hope;I Virtuosi Italiani,79,0.003448,ambient
25884,Bfg Division,Mick Gordon,66,0.003511,electronic
38408,Fight Fire With Fire - Remastered,Metallica,61,0.004354,hard-rock
31738,Light of the Seven,Ramin Djawadi,61,0.004606,german


## Em quais gêneros o modelo mais erra?

Se os erros se concentram em gêneros específicos, o modelo pode estar enviesado. Isso é informação direta pra quem for usar o app.

In [7]:
errors = results[results["tipo"].isin(["falso_positivo", "falso_negativo"])]
error_by_genre = errors["track_genre"].value_counts().head(10).reset_index()
error_by_genre.columns = ["genre", "errors"]

fig = px.bar(
    error_by_genre, x="errors", y="genre",
    orientation="h", text="errors",
    title="Top 10 gêneros com mais erros",
    labels={"errors": "Quantidade de erros", "genre": ""},
    color_discrete_sequence=["#e74c3c"],
)
fig.update_layout(height=400, template="plotly_white", yaxis=dict(autorange="reversed"))
fig.show()

## Scatter: probabilidade prevista vs popularidade real

In [8]:
color_map = {
    "acerto_hit": "#2ecc71",
    "acerto_nao_hit": "#3498db",
    "falso_positivo": "#e74c3c",
    "falso_negativo": "#f39c12",
}

sample = results.sample(n=min(3000, len(results)), random_state=42)

fig = px.scatter(
    sample, x="prob", y="popularity",
    color="tipo", color_discrete_map=color_map,
    hover_data=["name", "artist", "track_genre"],
    title="Probabilidade prevista vs popularidade real",
    labels={"prob": "Probabilidade prevista de hit", "popularity": "Popularity (Spotify)"},
    opacity=0.5,
)
fig.add_hline(y=60, line_dash="dash", line_color="gray", annotation_text="Threshold hit (60)")
fig.add_vline(x=0.5, line_dash="dash", line_color="gray", annotation_text="Threshold modelo (0.5)")
fig.update_layout(height=550, template="plotly_white")
fig.show()

## Distribuição da probabilidade por tipo de resultado

In [9]:
fig = px.histogram(
    results, x="prob", color="tipo",
    color_discrete_map=color_map,
    nbins=50, barmode="overlay", opacity=0.6,
    title="Distribuição da probabilidade prevista por tipo de resultado",
    labels={"prob": "Probabilidade prevista", "count": "Quantidade"},
)
fig.update_layout(height=400, template="plotly_white")
fig.show()

## Conclusões

**Em qual tipo de música o modelo é menos confiável?**

Músicas com popularidade entre 45 e 70 são a zona cinzenta. Elas têm perfil sonoro parecido com hits (danceability alta, loudness competitivo) mas não chegaram lá — ou são hits que soam diferentes do padrão (acústicos, instrumentais, lentos). Os erros também se concentram em gêneros de nicho, onde o modelo tem menos exemplos pra aprender.

**O que isso significa pra quem for usar o app?**

O modelo funciona como filtro inicial, não como decisão final. Se diz "alto potencial", a música provavelmente tem o perfil sonoro certo — vale investigar. Se diz "baixo potencial", não descarte direto. Pergunte o que mais a música tem a favor: o artista já tem base de fãs? Tem potencial de viralizar? O timing é bom?

O app não ouve a música. Ele lê números. Tudo que não está nos números — letra, carisma do artista, marketing, sorte — escapa do modelo.